# Build portable RAG index tren Google Colab

Notebook nay build index cho pipeline hien tai trong `src/backend/pipeline.py`:

`token chunks -> E5 embeddings -> Qdrant dense index + BM25 index -> smoke test -> zip artifact`

Artifact khong chua API key va khong chua model cache. Sau khi tai artifact ve, giai nen vao thu muc goc project local.

In [ ]:
# Cell 1 - Cau hinh repo va du lieu
from pathlib import Path

REPO_URL = "https://github.com/TiiAyyLuvBear/Text-Mining---RAG-on-News.git"
BRANCH = "test_feature_ta"
REPO_DIR = Path("/content/Text-Mining---RAG-on-News")

# Khuyen nghi: dat corpus tren Google Drive de tranh upload lai moi lan.
USE_DRIVE = True
DRIVE_CORPUS = Path("/content/drive/MyDrive/rag_data/vieonline_news_chunks_token.jsonl")

# Dung full corpus. Neu chi smoke test, dat LIMIT = 1000.
LIMIT = None
BATCH_SIZE = 64
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
COLLECTION = "news_bge_token"
print("Repo:", REPO_DIR)
print("Drive corpus:", DRIVE_CORPUS)
print("LIMIT:", LIMIT, "BATCH_SIZE:", BATCH_SIZE)

In [ ]:
# Cell 2 - Bat GPU Colab, clone repo va cai dependency
from pathlib import Path
import os, sys, subprocess

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Repo da ton tai:", REPO_DIR)

%cd $REPO_DIR
%pip install -q qdrant-client rank-bm25 sentence-transformers transformers accelerate python-dotenv
sys.path.insert(0, str(REPO_DIR))
print("Python:", sys.version)
print("Working directory:", Path.cwd())

In [ ]:
# Cell 3 - Mount Google Drive va kiem tra GPU
import torch

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Colab chua bat GPU. Vao Runtime > Change runtime type > T4 GPU roi chay lai.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 4 - Chuan bi corpus chunk
import json, shutil
from pathlib import Path

CHUNK_PATH = REPO_DIR / "data/chunking/output/vieonline_news_chunks_token.jsonl"
if USE_DRIVE and DRIVE_CORPUS.is_file():
    CHUNK_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not CHUNK_PATH.exists() or CHUNK_PATH.stat().st_size != DRIVE_CORPUS.stat().st_size:
        print("Copy corpus tu Drive vao Colab...")
        shutil.copy2(DRIVE_CORPUS, CHUNK_PATH)
elif not CHUNK_PATH.is_file():
    print("Khong tim thay corpus trong repo/Drive. Chon file JSONL tu may.")
    from google.colab import files
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    CHUNK_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(Path(uploaded_name), CHUNK_PATH)

if not CHUNK_PATH.is_file():
    raise FileNotFoundError(CHUNK_PATH)
print("Corpus:", CHUNK_PATH)
print("Size MB:", round(CHUNK_PATH.stat().st_size / 1024**2, 2))

count = 0
first_rows = []
with CHUNK_PATH.open(encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue
        count += 1
        if len(first_rows) < 2:
            first_rows.append(json.loads(line))
print("Non-empty chunks:", count)
for row in first_rows:
    print({"chunk_id": row.get("chunk_id"), "article_id": row.get("article_id"), "text_preview": str(row.get("text", ""))[:160]})
if count == 0:
    raise ValueError("Corpus khong co chunk co text.")

In [ ]:
# Cell 5 - Dat env truoc khi import src.backend.config/pipeline
import os

os.environ["HF_HOME"] = "/content/huggingface_cache"
os.environ["NEWS_CHUNK_PATH"] = str(CHUNK_PATH.relative_to(REPO_DIR)).replace("\\", "/")
os.environ["QDRANT_PATH"] = "data/qdrant_news"
os.environ["QDRANT_COLLECTION"] = COLLECTION
os.environ["BM25_INDEX_PATH"] = "data/qdrant_news_bm25.pkl"
os.environ["EMBEDDING_MODEL"] = EMBEDDING_MODEL
os.environ["RERANKER_MODEL"] = RERANKER_MODEL
os.environ["MODEL_DEVICE"] = "cuda:0"
os.environ["EMBEDDING_DEVICE"] = "cuda:0"
os.environ["RERANKER_DEVICE"] = "cuda:0"
os.environ["MODEL_DTYPE"] = "float16"
os.environ["LLM_PROVIDER"] = "api"
os.environ["REQUEST_TRACE_ENABLED"] = "false"

from src.backend.pipeline import NewsPipeline
from src.backend import config
print("CHUNK_PATH:", config.CHUNK_PATH)
print("QDRANT_PATH:", config.QDRANT_PATH)
print("BM25_INDEX_PATH:", config.BM25_INDEX_PATH)

In [ ]:
# Cell 6 - Build dense Qdrant index va BM25 index
# Neu GPU het VRAM, giam BATCH_SIZE xuong 32 hoac 16 o Cell 1.
pipeline = NewsPipeline()
try:
    indexed = pipeline.build_index(batch_size=BATCH_SIZE, limit=LIMIT)
    stored = pipeline.client.count(collection_name=config.COLLECTION, exact=True).count
    print("Indexed chunks:", indexed)
    print("Qdrant points:", stored)
    assert indexed == stored, (indexed, stored)
finally:
    pipeline.close()

assert config.BM25_INDEX_PATH.is_file(), config.BM25_INDEX_PATH
print("BM25 size MB:", round(config.BM25_INDEX_PATH.stat().st_size / 1024**2, 2))

In [ ]:
# Cell 7 - Smoke test dense + BM25 retrieval
pipeline = NewsPipeline()
smoke_questions = [
    "Tại sao nước dùng hầm xương hoặc nước lẩu có thể gây hại cho thận?",
    "Ai là đạo diễn của bộ phim truyền hình Đi về phía lửa?",
    "So sánh doanh thu của FPT và Hòa Phát trong năm 2023.",
]
try:
    for question in smoke_questions:
        rows = pipeline.retrieve(question, limit=5)
        print("\nQuestion:", question)
        print("Top results:")
        for row in rows[:5]:
            print({"article_id": row.get("article_id"), "chunk_id": row.get("chunk_id"), "score": round(float(row.get("retrieval_score", 0)), 4), "title": row.get("title")})
        assert rows, "Retrieval khong tra ve ket qua"
finally:
    pipeline.close()

In [ ]:
# Cell 8 - Tao manifest va zip artifact
import datetime as dt, json, shutil
from pathlib import Path

artifact_root = Path("/content/rag_index_artifact")
zip_path = Path("/content/rag_colab_data.zip")
if artifact_root.exists():
    shutil.rmtree(artifact_root)
if zip_path.exists():
    zip_path.unlink()
(artifact_root / "data").mkdir(parents=True)
shutil.copytree(config.QDRANT_PATH, artifact_root / "data" / config.QDRANT_PATH.name, ignore=shutil.ignore_patterns(".lock"))
shutil.copy2(config.BM25_INDEX_PATH, artifact_root / "data" / config.BM25_INDEX_PATH.name)
manifest = {
    "created_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "chunk_count": indexed,
    "embedding_model": config.EMBEDDING_MODEL,
    "reranker_model": config.RERANKER_MODEL,
    "qdrant_collection": config.COLLECTION,
    "qdrant_path": "data/qdrant_news",
    "bm25_path": "data/qdrant_news_bm25.pkl",
    "note": "Copy data/qdrant_news and data/qdrant_news_bm25.pkl into local project data/. Do not copy .lock.",
}
(artifact_root / "index_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
(artifact_root / "README.txt").write_text("Giai nen vao thu muc goc project. Artifact co data/qdrant_news va data/qdrant_news_bm25.pkl.", encoding="utf-8")
shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=artifact_root)
print("Artifact:", zip_path)
print("Size MB:", round(zip_path.stat().st_size / 1024**2, 2))
print("Manifest:", manifest)

In [ ]:
# Cell 9 - Tai zip ve may
from google.colab import files
files.download("/content/rag_colab_data.zip")

## Cai artifact tren local

Dat file zip tai thu muc goc project, sau do chay trong PowerShell:

```powershell
Expand-Archive -Path .\rag_colab_data.zip -DestinationPath . -Force
Invoke-RestMethod http://127.0.0.1:8000/api/health
```

Ket qua health phai co `index_ready: True`. Local khong can chay lai `build_index`. Phai giu cac gia tri `QDRANT_COLLECTION`, `EMBEDDING_MODEL` va `RERANKER_MODEL` giong manifest.